# Heart Disease Prediction — Step 1: Data Cleaning

**Dataset:** merged heart disease dataset (5 publicly available sources), sourced from Kaggle.

This notebook covers the **cleaning** stage of the pipeline:
1. Load the raw data
2. Fix structural issues in the raw file
3. Handle missing values
4. Remove duplicate records
5. Save a clean dataset to use for exploration (next notebook)

No modelling happens here yet — the goal is just a dataset we can trust.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## 1. Load the raw data

We load the file exactly as it was given to us, before touching anything.

In [2]:
raw = pd.read_excel('heart_disease_raw.xlsx')
print("Shape:", raw.shape)
raw.head()

Shape: (2181, 1)


,"age,sex,cp,trestbps,chol,fbs,restecg,thalachh,exang,oldpeak,slope,ca,thal,target"
0,"63,1,3,145,233,1,0,150,0,2.3,0,0,1,1"
1,"37,1,2,130,250,0,1,187,0,3.5,0,0,2,1"
2,"41,0,1,130,204,0,0,172,0,1.4,2,0,2,1"
3,"56,1,1,120,236,0,1,178,0,0.8,2,0,2,1"
4,"57,0,0,120,354,0,1,163,1,0.6,2,0,2,1"


## 2. Fix a structural issue in the raw file

When we loaded the file, all 14 fields came through squashed into **one column**, separated by commas
(the column name itself was `"age,sex,cp,trestbps,..."`). This is a formatting issue with how the
file was saved — not something wrong with the data itself.

We split that single column into the 14 separate columns it's supposed to have.

In [3]:
df = raw.copy()

if df.shape[1] == 1:
    col = df.columns[0]
    df = df[col].str.split(',', expand=True)
    df.columns = col.split(',')

print("Shape after fixing columns:", df.shape)
df.head()

Shape after fixing columns: (2181, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalachh,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


## 3. Handle missing values

Missing values in this dataset are written as the text `"?"` instead of being left blank.
Pandas doesn't recognise `"?"` as "missing" automatically, so first we need to tell it to.

**Why this matters:** as long as `"?"` sits in a numeric column, pandas treats the *whole column*
as text — which means we can't do any maths (averages, correlations, model training) on it yet.

In [4]:
# Replace the '?' placeholder with a proper missing value marker
df = df.replace('?', np.nan)

# Convert every column to numeric now that '?' is out of the way
for column in df.columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

df.dtypes

age           int64
sex           int64
cp            int64
trestbps    float64
chol        float64
fbs         float64
restecg     float64
thalachh    float64
exang       float64
oldpeak     float64
slope       float64
ca          float64
thal        float64
target        int64
dtype: object

In [5]:
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df) * 100).round(1)

missing_summary = pd.DataFrame({'missing_count': missing_counts, 'missing_%': missing_pct})
missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_count', ascending=False)

,missing_count,missing_%
ca,291,13.3
thal,266,12.2
slope,190,8.7
chol,23,1.1
fbs,8,0.4
trestbps,1,0.0
restecg,1,0.0
thalachh,1,0.0
exang,1,0.0


**What we're seeing:** a few columns (`ca`, `thal`, `slope`) are missing a *lot* of values —
this is because not every one of the 5 source hospitals recorded those particular tests. Other
columns are only missing 1–22 values, which is minor.

**How we'll fix it:**
- For **continuous** columns (`age`, `trestbps`, `chol`, `thalachh`, `oldpeak`) — fill missing values
  with the **median**. The median is a safe choice because it isn't thrown off by outliers.
- For **categorical** columns (`sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `ca`, `thal`) — fill
  missing values with the **mode** (the most frequently occurring value), since these represent
  categories, not a scale of numbers.

In [6]:
continuous_cols = ['age', 'trestbps', 'chol', 'thalachh', 'oldpeak']
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

for c in continuous_cols:
    df[c] = df[c].fillna(df[c].median())

for c in categorical_cols:
    df[c] = df[c].fillna(df[c].mode()[0])

print("Remaining missing values:", df.isnull().sum().sum())

Remaining missing values: 0


## 4. Remove duplicate records

Some rows in the merged file are **exact duplicates** — the same patient record appearing more than
once. Left in, these would let the same patient show up in both the training and test sets later,
which makes a model look more accurate than it really is.

In [7]:
before = df.shape[0]
df = df.drop_duplicates().reset_index(drop=True)
after = df.shape[0]

print(f"Removed {before - after} duplicate rows")
print(f"Remaining: {after} unique patient records")

Removed 1287 duplicate rows
Remaining: 894 unique patient records


## 5. Quick sanity check

Before saving, a last look to confirm everything is in order: no missing values, no duplicates,
sensible-looking ranges, and a roughly balanced target.

In [8]:
print("Final shape:", df.shape)
print("\nMissing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("\nTarget class balance:")
print(df['target'].value_counts())
print("\nSummary statistics:")
df.describe()

Final shape: (894, 14)

Missing values: 0
Duplicate rows: 0

Target class balance:
target
0    497
1    397
Name: count, dtype: int64

Summary statistics:


,age,sex,cp,trestbps,chol,fbs,restecg,thalachh,exang,oldpeak,slope,ca,thal,target
count,894.000000,894.000000,894.000000,894.000000,894.000000,894.000000,894.000000,894.000000,894.000000,894.000000,894.000000,894.00000,894.000000,894.000000
mean,52.302013,0.697987,1.665548,131.949664,249.043624,0.124161,0.572707,145.994407,0.325503,0.913758,1.633110,0.47651,3.042506,0.444072
std,9.205776,0.459388,1.126391,17.527605,56.342809,0.329950,0.757770,23.700468,0.468825,1.105398,0.573451,0.87199,1.709726,0.497140
min,28.000000,0.000000,0.000000,92.000000,85.000000,0.000000,0.000000,71.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
25%,45.000000,0.000000,1.000000,120.000000,212.000000,0.000000,0.000000,130.000000,0.000000,0.000000,1.000000,0.00000,2.000000,0.000000
50%,53.000000,1.000000,2.000000,130.000000,242.500000,0.000000,0.000000,150.000000,0.000000,0.500000,2.000000,0.00000,2.000000,0.000000
75%,59.000000,1.000000,3.000000,140.000000,277.000000,0.000000,1.000000,163.000000,1.000000,1.500000,2.000000,1.00000,3.000000,1.000000
max,77.000000,1.000000,3.000000,200.000000,603.000000,1.000000,2.000000,202.000000,1.000000,6.200000,3.000000,4.00000,7.000000,1.000000


## 6. Save the cleaned dataset

This cleaned file is what we'll use for exploration (EDA) in the next notebook.

In [9]:
df.to_csv('heart_disease_cleaned.csv', index=False)
print("Saved: heart_disease_cleaned.csv")

Saved: heart_disease_cleaned.csv
